# 02 — Elo, Form, and Head-to-Head Features

Computes, for every match in the full ATP/WTA history (not just Slams — Elo
and form need continuity across a player's whole career), the PRE-match state
for both players:

- **Overall Elo** and **surface-specific Elo** (experience-adjusted K-factor,
  same form FiveThirtyEight uses for tennis: newer players' ratings move
  faster, established players' are more stable), with a **margin-of-victory
  multiplier** on top (also adapted from FiveThirtyEight, originally for
  NBA/NFL Elo): a 6-0 6-0 6-0 sweep moves ratings more than a 7-6 7-6 7-6 win
  of the same match, and the multiplier is dampened by how surprising the
  result already was, so an already-big-favorite's expected blowout counts
  for less than a similarly dominant win by the underdog
- **Recent form**: win rate over their last 20 matches
- **Current streak**: consecutive wins (positive) or losses (negative)
  entering the match
- **Head-to-head**: prior meetings between these two players, all-time

ATP and WTA are treated as fully independent player pools throughout. This
is a strictly sequential computation — every match's features depend on
everything that happened before it for those two players — so it's a single
chronological pass rather than a vectorized operation.


In [1]:
import sys
sys.path.insert(0, '../src')

import pandas as pd
import numpy as np
import features as feat

pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 140)


## 1. Chronological ordering

Matches are sorted by tour → date → round → match number, so that (for
example) a player's quarterfinal result is reflected in their Elo before
their semifinal is processed — both share the same `tourney_date` (the
Monday of tournament week), so date alone isn't enough.


In [2]:
matches = pd.read_parquet('../data/processed/matches_all.parquet')
chron = feat.prepare_chronological(matches)
print(chron.shape)
chron[['tour', 'tourney_date', 'round', 'match_num', 'winner_name', 'loser_name']].head(3)


(361571, 50)


,tour,tourney_date,round,match_num,winner_name,loser_name
0,ATP,1967-12-28,QF,101,Alex Metreveli,Jan Kodes
1,ATP,1967-12-28,QF,101,Ion Tiriac,Zeljko Franulovic
2,ATP,1967-12-28,QF,102,Jaidip Mukerjea,Jan Kukal


## 2. Compute features

Walkovers (`score == 'W/O'`, no play occurred) are skipped for rating/form/H2H
*updates* but still get a pre-match feature value looked up — they carry no
information about relative skill, so including them in updates would be
noise.


In [3]:
%%time
featured = feat.build_temporal_features(chron, form_window=20)
print(featured.shape)


(361571, 60)
CPU times: user 8.18 s, sys: 189 ms, total: 8.37 s
Wall time: 8.41 s


In [4]:
featured.to_parquet('../data/processed/matches_with_features.parquet', index=False)
print('saved matches_with_features.parquet')


saved matches_with_features.parquet


## 3. Sanity checks

### Elo trajectory for a well-known player

Should start at the 1500 base rating on debut and climb through their peak
years.


In [5]:
def elo_trajectory(df, player_name):
    is_winner = df['winner_name'] == player_name
    is_loser = df['loser_name'] == player_name
    sub = df[is_winner | is_loser].copy()
    sub['elo'] = np.where(is_winner[is_winner | is_loser], sub['winner_elo_pre'], sub['loser_elo_pre'])
    return sub[['tourney_date', 'tourney_name', 'round', 'elo']]

traj = elo_trajectory(featured, 'Novak Djokovic')
traj.iloc[[0, len(traj)//4, len(traj)//2, 3*len(traj)//4, -1]]


,tourney_date,tourney_name,round,elo
134219,2004-04-09,Davis Cup G2 R1: SCG vs LAT,RR,1500.000000
152841,2010-02-08,Rotterdam,R16,2574.225520
165972,2014-05-26,Roland Garros,R16,2715.724766
181125,2019-08-26,US Open,R128,2536.390141
199369,2026-05-25,Roland Garros,R32,2447.182723


### Elo leaderboard

`current_ratings` gives each player's rating as of THEIR OWN last match in
the dataset — not a single as-of-today snapshot. So an all-time "best final
Elo ever reached" view mixes retired legends (whose last rating is near
their retirement peak) with active players (whose rating reflects however
they're doing right now). Both views below, to make that explicit.


In [6]:
current = feat.current_ratings(featured)
current.to_parquet('../data/processed/current_ratings.parquet', index=False)

elo_cols = ['player_name', 'overall_elo', 'last_match_date'] + [c for c in current.columns if c.startswith('elo_')]

print('ATP — best final Elo ever reached (all-time, includes retired players):')
display(current[current['tour'] == 'ATP'].sort_values('overall_elo', ascending=False).head(10)[elo_cols])

print('ATP — top 10 among players active in the last 2 years:')
active = current[current['last_match_date'] >= current['last_match_date'].max() - pd.Timedelta(days=730)]
display(active[active['tour'] == 'ATP'].sort_values('overall_elo', ascending=False).head(10)[elo_cols])

print('WTA — top 10 among players active in the last 2 years:')
display(active[active['tour'] == 'WTA'].sort_values('overall_elo', ascending=False).head(10)[elo_cols])


ATP — best final Elo ever reached (all-time, includes retired players):


,player_name,overall_elo,last_match_date,elo_Carpet,elo_Clay,elo_Grass,elo_Hard
15364,Jannik Sinner,2734.042175,2026-05-25,NaN,2588.772492,2095.911027,2595.332228
15206,Carlos Alcaraz,2593.046560,2026-04-13,NaN,2551.103949,2410.600053,2489.582107
15407,Novak Djokovic,2447.182723,2026-05-25,1989.870460,2341.742524,2404.358047,2395.145971
13864,Roger Federer,2445.924437,2021-06-28,2047.075947,2331.405050,2295.524879,2412.362963
15237,Arthur Fils,2386.562530,2026-05-06,NaN,2353.950607,1772.245535,2222.713465
15355,Alexander Zverev,2386.381079,2026-05-25,NaN,2294.660307,1986.856118,2297.413685
13948,Juan Martin del Potro,2376.749302,2022-02-07,1697.642175,2240.679898,1986.435131,2307.417600
12290,Robin Soderling,2375.816495,2011-07-11,1842.206511,2362.092909,2066.517340,2331.841043
15423,Rafael Jodar,2363.734482,2026-05-25,NaN,2267.565963,NaN,1934.673980
10375,Patrick Rafter,2342.473780,2001-11-30,1962.619115,2078.918753,2189.378585,2272.562446


ATP — top 10 among players active in the last 2 years:


,player_name,overall_elo,last_match_date,elo_Carpet,elo_Clay,elo_Grass,elo_Hard
15364,Jannik Sinner,2734.042175,2026-05-25,NaN,2588.772492,2095.911027,2595.332228
15206,Carlos Alcaraz,2593.046560,2026-04-13,NaN,2551.103949,2410.600053,2489.582107
15407,Novak Djokovic,2447.182723,2026-05-25,1989.870460,2341.742524,2404.358047,2395.145971
15237,Arthur Fils,2386.562530,2026-05-06,NaN,2353.950607,1772.245535,2222.713465
15355,Alexander Zverev,2386.381079,2026-05-25,NaN,2294.660307,1986.856118,2297.413685
15423,Rafael Jodar,2363.734482,2026-05-25,NaN,2267.565963,NaN,1934.673980
14465,Rafael Nadal,2331.675171,2024-11-19,1464.318895,2407.734277,2194.290326,2230.722845
15422,Joao Fonseca,2328.555106,2026-05-25,NaN,2227.266081,1592.725037,2173.479000
15451,Daniil Medvedev,2305.593731,2026-05-25,NaN,2114.039322,2018.059158,2299.516749
15417,Casper Ruud,2291.971018,2026-05-25,NaN,2272.752855,1519.819438,2065.195532


WTA — top 10 among players active in the last 2 years:


,player_name,overall_elo,last_match_date,elo_Carpet,elo_Clay,elo_Grass,elo_Hard
15501,Aryna Sabalenka,2650.782801,2026-05-25,NaN,2377.766923,2041.520609,2561.084885
15356,Elena Rybakina,2598.501063,2026-05-25,NaN,2305.542358,2028.081898,2547.247764
15513,Marta Kostyuk,2507.220265,2026-05-25,NaN,2355.614848,1611.638023,2263.684660
15273,Jessica Pegula,2467.222024,2026-05-25,NaN,2153.623441,1990.627050,2401.813618
15457,Iga Swiatek,2449.820123,2026-05-25,NaN,2346.252295,1983.111089,2295.029670
15507,Mirra Andreeva,2449.516735,2026-05-25,NaN,2303.064137,1873.887459,2263.202854
15503,Elina Svitolina,2443.169102,2026-05-25,1711.198699,2308.657615,1934.883445,2319.067693
15465,Coco Gauff,2434.233271,2026-05-25,NaN,2358.092519,1758.829135,2299.885660
15454,Karolina Muchova,2408.575134,2026-05-25,NaN,2100.645676,1626.487185,2351.119250
15514,Maja Chwalinska,2381.593957,2026-05-25,NaN,2210.744029,1500.000000,1459.408535


### Does Elo actually predict anything?

Quick check before moving to modeling: if we just picked the player with the
higher pre-match overall Elo as the predicted winner, what accuracy do we
get? This isn't the real model (no surface/form/H2H blending, no
calibration) — it's a floor the real model needs to beat.


In [7]:
valid = featured[(featured['score'] != 'W/O')].copy()
valid['elo_favorite_correct'] = valid['winner_elo_pre'] > valid['loser_elo_pre']

print(f"All-time: {valid['elo_favorite_correct'].mean():.3f} accuracy over {len(valid):,} matches")

recent = valid[valid['tourney_date'] >= '2015-01-01']
print(f"2015+ only: {recent['elo_favorite_correct'].mean():.3f} accuracy over {len(recent):,} matches")

slams_recent = recent[recent['tourney_level'] == 'G']
print(f"2015+ Slams only: {slams_recent['elo_favorite_correct'].mean():.3f} accuracy over {len(slams_recent):,} matches")


All-time: 0.686 accuracy over 359,016 matches
2015+ only: 0.653 accuracy over 61,654 matches
2015+ Slams only: 0.708 accuracy over 11,385 matches


### Does the margin-of-victory multiplier actually help?

Re-running the exact same pass with `use_margin_of_victory=False` isolates
its effect on the 2022+ Grand Slam test window (the same split the win
classifier uses) — a direct ablation rather than just asserting the idea is
sound.


In [8]:
no_mov = feat.build_temporal_features(chron, form_window=20, use_margin_of_victory=False)
test_window = lambda df: df[(df['tourney_level'] == 'G') & (df['tourney_date'] >= '2022-01-01') & (df['score'] != 'W/O')]

for tour in ['ATP', 'WTA']:
    base_acc = (test_window(no_mov[no_mov['tour'] == tour]).pipe(lambda d: d['winner_elo_pre'] > d['loser_elo_pre']).mean())
    mov_acc = (test_window(featured[featured['tour'] == tour]).pipe(lambda d: d['winner_elo_pre'] > d['loser_elo_pre']).mean())
    print(f"{tour}: {base_acc:.3f} without MOV -> {mov_acc:.3f} with MOV")


ATP: 0.707 without MOV -> 0.714 with MOV
WTA: 0.685 without MOV -> 0.693 with MOV


A small but consistent gain for both tours — kept on by default.


### Form, streak, and H2H distributions


In [9]:
print('Form win-rate (last 20 matches) distribution:')
display(featured['winner_form_winrate'].describe())

print('Win streak distribution (winners):')
display(featured['winner_win_streak'].describe())

print('H2H: fraction of matches with at least one prior meeting:')
print((featured['winner_h2h_wins'] + featured['winner_h2h_losses'] > 0).mean())


Form win-rate (last 20 matches) distribution:


count    357331.000000
mean          0.550209
std           0.186130
min           0.000000
25%           0.444444
50%           0.550000
75%           0.666667
max           1.000000
Name: winner_form_winrate, dtype: float64

Win streak distribution (winners):


count    361571.000000
mean          0.419138
std           3.454182
min         -29.000000
25%          -1.000000
50%           1.000000
75%           2.000000
max          73.000000
Name: winner_win_streak, dtype: float64

H2H: fraction of matches with at least one prior meeting:
0.4144497208017236


## Summary

`data/processed/matches_with_features.parquet` now carries, for every
historical match: `{winner,loser}_elo_pre`, `{winner,loser}_surface_elo_pre`,
`{winner,loser}_form_winrate`, `{winner,loser}_win_streak`, and
`winner_h2h_wins`/`winner_h2h_losses`. `current_ratings.parquet` has the
latest overall + per-surface Elo for every active/historical player.

**Next**: the win classifier notebook filters this down to Grand Slam
matches, adds the remaining planned features (seed, round, rest days,
handedness, style features from notebook 01), and does a proper chronological
train/test split.
